# Analyzing Data Science Task to LMs

In [1]:
# imports
import yaml
import json
from stat_genie.blade_pipeline.baselines.config import MultiRunConfig
from stat_genie.blade_pipeline.baselines.multirun import multirun_llm
from stat_genie.blade_pipeline.additions.prompt.prompt import PromptGenerator
import os
from os.path import join
from pathlib import Path
from stat_genie.blade_pipeline.additions.analysis.get_model_output import get_model_output
from stat_genie.blade_pipeline.additions.perturbations.feature_names import FeaturePerturbation
from stat_genie.blade_pipeline.additions.analysis.fix_code import check_and_fix_code
import importlib.util
import sys
import pandas as pd
from stat_genie.blade_pipeline.baselines.multirun import _format_cvars_for_prompt
from stat_genie.blade_pipeline.additions.analysis.conclusion import write_final_answer_code, make_conclusion

In [2]:
### set config parameters

# set up config object
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_config = yaml.safe_load(open("../../config/llm_config.yml"))
llm_config["provider"] = llm_provider
llm_config["model"] = llm_model
llm_eval_config = llm_config

# set rest of parameters
output_dir = "analysis1_output"
run_dataset = "hurricane"
use_agent = False
use_data_desc = True
num_runs=3
use_code_cache=False

In [3]:
# the MultiRunConfig object is how BLADE standardizes experiment configuration
single_run_config = MultiRunConfig(llm=llm_config,
                llm_eval=llm_eval_config,
                output_dir=output_dir,
                run_dataset=run_dataset,
                use_agent=use_agent,
                use_data_desc=use_data_desc,
                num_runs=num_runs,
                use_code_cache=use_code_cache,
                fix_code=True,
)

In [4]:
# prevent cache use
single_run_config.llm.use_cache = False
single_run_config.llm_eval.use_cache = False

In [5]:
### run the experiment
# version #1 of the analysis does NOT apply any feature perturbation.
multirun_llm(single_run_config)

[2025-12-04 10:42:32.91][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:42:33.68][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:42:34.27][llm.py:109 - stat_genie.blade_pipeline.llms.llm:generate][PROMPT] Sending prompt from <class 'stat_genie.blade_pipeline.baselines.lm.gen_analysis.GenAnalysisLM'>
===================[[system]]===================
You are an AI Data Analysis Assistant who is an expert at writing an end-to-end scientific analysis given a research question and a dataset. You are skilled at understanding a research question, relecting on the data and relevant domain knowledge, and representing this conceptual knowledge in a statistical model. Key to this modeling process is formalizing the conceptual model, which inc

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[2025-12-04 10:45:39.63][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:45:40.12][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 10:46:18.61][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  38.48 seconds
[2025-12-04 10:46:18.61][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Max

[2025-12-04 10:46:20.22][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:46:20.51][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 10:47:19.76][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  59.25 seconds
[2025-12-04 10:47:19.77][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


...

[2025-12-04 10:51:25.44][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:51:25.66][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 10:52:13.77][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  48.12 seconds
[2025-12-04 10:52:13.78][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 10:52:13.92][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:52:14.20][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 10:53:08.38][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  54.18 

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[2025-12-04 10:53:16.84][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:53:17.08][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 10:54:14.39][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  57.30 seconds
[2025-12-04 10:54:14.40][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 10:54:14.43][multirun.py:236 - stat_genie.blade_pipeline.baselines.multirun:__save_results][INFO] Wrote final answer extraction code for analysis 1. It required 0 iteration(s) to be correct.
[2025-12-04 10:54:14.57][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:54:14.88][base.py:60 - stat_gen

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[2025-12-04 10:54:19.93][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:54:20.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 10:54:51.34][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  31.02 seconds
[2025-12-04 10:54:51.35][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 10:54:51.45][multirun.py:236 - stat_genie.blade_pipeline.baselines.multirun:__save_results][INFO] Wrote final answer extraction code for analysis 2. It required 0 iteration(s) to be correct.
[2025-12-04 10:54:51.70][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 10:54:52.87][base.py:60 - stat_gen